# 📈 EDA Senior – Events
**Gráficas:** 11 | **Dataset:** `events_clean.csv` / `events.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, seaborn as sns, warnings
warnings.filterwarnings('ignore')
from src.config import *
from src.data_cleaning import load_and_clean_events, summarize_dataframe, calculate_sparsity, cold_start_analysis, conversion_funnel
from src.viz import setup_style, plot_funnel, plot_lorenz, plot_time_heatmap, plot_cold_start_pie, plot_long_tail
setup_style()

path = EVENTS_CLEAN if os.path.exists(EVENTS_CLEAN) else EVENTS_RAW
print(f'Cargando: {path}')
df = load_and_clean_events(path)
summarize_dataframe(df, 'Events')

---
## 📊 Gráfica 1 – Distribución de Tipos de Evento

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
counts = df['event'].value_counts()
bars = sns.barplot(x=counts.index, y=counts.values, palette=['#2ecc71','#f39c12','#e74c3c'], ax=ax)
ax.set_yscale('log')
for bar, v in zip(ax.patches, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.15, f'{v:,}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Distribución de Tipos de Evento (escala log)', fontweight='bold', fontsize=14)
ax.set_xlabel('Tipo de evento'); ax.set_ylabel('Conteo (log)')
plt.tight_layout(); plt.show()

## 📊 Gráfica 2 – Embudo de Conversión

In [ ]:
funnel = conversion_funnel(df)
display(funnel)
plot_funnel(funnel, title='Embudo de Conversión – NexusDataCo')

## 📊 Gráfica 3 – Interacciones por Hora (Stacked Bar)

In [ ]:
df['hour'] = df['datetime'].dt.hour
df['dow']  = df['datetime'].dt.day_name()
df['date'] = df['datetime'].dt.date

fig, ax = plt.subplots(figsize=(14,5))
hourly = df.groupby(['hour','event']).size().unstack(fill_value=0)
hourly.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', width=0.85)
ax.set_title('Interacciones por Hora del Día (Stacked)', fontweight='bold')
ax.set_xlabel('Hora'); ax.set_ylabel('Interacciones')
ax.legend(title='Evento', loc='upper left'); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 📊 Gráfica 4 – Heatmap Actividad Semanal

In [ ]:
plot_time_heatmap(df, title='Heatmap de Actividad – Hora × Día de la Semana')

## 📊 Gráfica 5 – Serie de Tiempo Diaria

In [ ]:
daily = df.groupby(['date','event']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(16,5))
daily.plot(ax=ax, colormap='Set1', lw=1.5)
ax.set_title('Evolución Diaria de Eventos (May–Sep 2015)', fontweight='bold')
ax.set_xlabel('Fecha'); ax.set_ylabel('Interacciones')
plt.tight_layout(); plt.show()

## 📊 Gráfica 6 – Sparsity de la Matriz

In [ ]:
sp = calculate_sparsity(df)
for k, v in sp.items(): print(f'{k:<18}: {v:>12,}' if isinstance(v,int) else f'{k:<18}: {v:>12.4%}')

fig, ax = plt.subplots(figsize=(7,5))
vals   = [1 - sp['sparsity'], sp['sparsity']]
labels = ['Interacciones existentes\n(0.0008%)', f"Celdas vacías\n(Sparsity: {sp['sparsity']:.4%})"]
ax.pie(vals, labels=labels, colors=['#2ecc71','#ecf0f1'],
       autopct=lambda p: f'{p:.4f}%' if p < 1 else f'{p:.2f}%',
       startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Sparsity de la Matriz Usuario–Ítem', fontweight='bold')
plt.tight_layout(); plt.show()

## 📊 Gráfica 7 – Segmentación Cold-Start (Pie)

In [ ]:
cs = cold_start_analysis(df)
for k, v in cs.items():
    if k != 'total_users': print(f'{k:<20}: {v:>9,} ({v/cs["total_users"]*100:.2f}%)')
plot_cold_start_pie(cs)

## 📊 Gráficas 8–9 – Distribución de Interacciones por Usuario

In [ ]:
# Serie de interacciones por usuario
user_inter = df.groupby('visitorid').size().rename('interactions')

fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Gráfica 8: Histograma (capped)
user_inter.clip(upper=30).value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='#9b59b6', edgecolor='white')
axes[0].set_title('Histograma Interacciones / Usuario (cap=30)', fontweight='bold')
axes[0].set_xlabel('Nº interacciones'); axes[0].set_ylabel('Usuarios')

# Gráfica 9: Violin por tipo de evento
user_event = df.groupby(['visitorid','event']).size().reset_index(name='cnt')
sns.violinplot(data=user_event, x='event', y='cnt', palette='Set2', cut=0, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Interacciones por Evento (Violin, escala log)', fontweight='bold')
axes[1].set_xlabel('Tipo de evento'); axes[1].set_ylabel('Interacciones (log)')
plt.tight_layout(); plt.show()

## 📊 Gráficas 10–11 – Long-Tail & Curva de Lorenz

In [ ]:
item_freq = df['itemid'].value_counts()
plot_long_tail(item_freq)

lt = (item_freq < LONG_TAIL_THRESHOLD).sum()
pop = (item_freq >= LONG_TAIL_THRESHOLD).sum()
print(f'Long-Tail (<{LONG_TAIL_THRESHOLD}):  {lt:,} ítems  ({lt/len(item_freq)*100:.1f}%)')
print(f'Populares (>={LONG_TAIL_THRESHOLD}): {pop:,} ítems ({pop/len(item_freq)*100:.1f}%)')

In [ ]:
gini = plot_lorenz(user_inter, label='Usuarios')
print(f'Coeficiente de Gini: {gini:.4f}  (0=igualdad, 1=máxima concentración)')

---
## ✅ Resumen Ejecutivo

| Métrica | Valor | Conclusión |
|---------|-------|------------|
| Sparsity | 99.9992% | CF puro inviable |
| Cold-Start (1 inter.) | 71.15% | Obligatorio CBF |
| Long-Tail (<5) | 61.3% | Diversidad > Popularidad |
| Conversión view→buy | 0.81% | Oportunidad de mejora |
| Gini interacciones | ~0.9 | Alta concentración |